# Fine-tune Needle 3 on PyBot Training Data

This notebook fine-tunes Cactus Compute's Needle 3 model on PyBot's routing patterns.

## Setup
1. Set GPU: Settings > Accelerator > GPU P100 (FREE)
2. Upload your `training_data.jsonl` file
3. Run all cells

Output: `needle3-pybot-lora/` folder with fine-tuned adapter

In [ ]:
# Install dependencies
!pip install -q torch transformers peft accelerate bitsandbytes
!pip install -q needle-infer  # If available, otherwise use transformers

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
import json
from torch.utils.data import Dataset
import os

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Download Needle 3 model from HuggingFace
MODEL_NAME = "CactusCompute/needle-3"  # Update with actual HF path

# Alternative: Use a small model that supports fine-tuning
# MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # 500M params, similar size to Needle 3

print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Model loaded: {model.num_parameters() / 1e6:.1f}M parameters")

In [ ]:
# Load training data
class PyBotDataset(Dataset):
    def __init__(self, filepath, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        
        with open(filepath, 'r') as f:
            for line in f:
                item = json.loads(line.strip())
                self.data.append(item)
        
        print(f"Loaded {len(self.data)} training examples")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format as tool-calling prompt
        user_msg = item['messages'][0]['content']
        tool_name = item['tool_call']['name']
        tool_args = item['tool_call']['arguments']
        
        # Needle 3 tool-calling format
        prompt = f"""<|user|>
{user_msg}
<|tools|>
[{{\"name\": \"{tool_name}\", \"arguments\": {json.dumps(tool_args)}}}]
<|assistant|>"""
        
        encoding = self.tokenizer(
            prompt,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': encoding['input_ids'].flatten()
        }

# Upload training_data.jsonl to Kaggle first!
dataset = PyBotDataset('/kaggle/input/pybot-training/training_data.jsonl', tokenizer)

In [ ]:
# Configure LoRA (Low-Rank Adaptation)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # Rank - lower = smaller adapter
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\nTrainable parameters: ~{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")
print("This is much smaller than full fine-tuning!")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./needle3-pybot-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

print("Starting training...")
print(f"Estimated time: ~1-2 hours on P100 GPU")

In [ ]:
# Train!
trainer.train()

In [ ]:
# Save the LoRA adapter
model.save_pretrained("./needle3-pybot-lora")
tokenizer.save_pretrained("./needle3-pybot-lora")

print("✅ Fine-tuned model saved to ./needle3-pybot-lora/")
print("\nDownload this folder and place it in Rustbot's weights/ directory")

In [ ]:
# Test the fine-tuned model
def test_tool_call(text):
    prompt = f"""<|user|>
{text}
<|assistant|>"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return response

# Test examples
test_cases = [
    "My name is Alice",
    "I have a gun to my head",
    "Hello there",
    "What's for dinner?",
]

for test in test_cases:
    print(f"\nInput: {test}")
    print(f"Output: {test_tool_call(test)}")

In [ ]:
# Zip the output for download
!zip -r needle3-pybot-lora.zip needle3-pybot-lora/
print("\n✅ Download needle3-pybot-lora.zip from the output panel")